In [ ]:
# import numpy as np
# obs = np.load("results/obs_1.npy")
# print(obs.shape)
# import matplotlib.pyplot as plt
# for i in range(obs.shape[0]):
#     plt.imshow(obs[i, :, :, :3]/255.)
#     plt.show()
#     plt.imsave(f"results/obs_{i}.png", obs[i, :, :, :3]/255.)

In [ ]:
%reload_ext autoreload
%autoreload 2
import sys
sys.path.append("..")

# check bev images
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import cv2

# data_file = "/home/junzhe_lighthouse/lighthouse/scratch/isaac_scenes_v1/episode_data/grCommercial_hospital/height_map.npz"
# data_file = "/home/junzhe_lighthouse/lighthouse/scratch/isaac_scenes_v1/episode_data/grCommercial_hospital/bev_map.npz"
# data_file = "/home/junzhe_lighthouse/lighthouse/scratch/isaac_scenes_v1/episode_data/grCommercial_superMarket/height_map.npz"
# data_file = "/home/junzhe_lighthouse/lighthouse/scratch/isaac_scenes_v1/episode_data/grCommercial_superMarket/bev_map.npz"
data_file = "/home/junzhe_lighthouse/lighthouse/scratch/isaac_scenes_v1/episode_data/vc_ny/bev_map.npz"
# data_file = "/home/junzhe_lighthouse/lighthouse/scratch/isaac_scenes_v1/episode_data/vc_ny/height_map.npz"
data = np.load(data_file)
# visualize the bev map with plotly
rgb, depth = data['rgb'], data['depth'][:,:,0]
depth[depth==np.inf] = 0
fig = px.imshow(rgb)
fig.show()

# fig = go.Figure(data=go.Heatmap(z=depth, colorscale='Viridis'))
fig = px.imshow(depth)
fig.show()

cv2.imwrite("rgb.png", rgb[...,[2,1,0,3]])


In [ ]:
import cv2


def generate_kernels():
    """
    Generate 4 different 3x3 kernels where each has one 1 and one -1 at opposite ends.

    Returns:
        np.ndarray: An array of shape (4, 3, 3) containing the four kernels.
    """
    kernels = np.zeros((4, 3, 3))  # Initialize 4 kernels with zeros
    
    positions = [((0, 0), (2, 2)),  # Top-left to bottom-right
                ((0, 2), (2, 0)),  # Top-right to bottom-left
                ((0, 1), (2, 1)),  # Top-center to bottom-center
                ((1, 0), (1, 2))]  # Middle-left to middle-right

    for i, (pos1, pos2) in enumerate(positions):
        kernels[i, pos1[0], pos1[1]] = 1   # Set 1 at one end
        # kernels[i, 1, 1] = 1   # Set 1 at one end
        kernels[i, pos2[0], pos2[1]] = -1  # Set -1 at the opposite end
    
    return kernels

local_w, local_h = depth.shape
height_map = np.ones((local_w, local_h))*(-1000.0)
height_map[depth>0] = depth[depth>0]

# hand desgined gradient kernel that has physical unit
kernels = generate_kernels()
filtered_img = []
for kernel in kernels:
    img = cv2.filter2D(src=height_map, ddepth=-1, kernel=kernel)
    filtered_img.append(img)
filtered_img = np.stack(filtered_img)  # Shape: (4, height, width)
# take the max across the 4 kernels to get the final gradient map
gradient_map = np.max(np.abs(filtered_img), axis=0)

# calculate obstacle map
obstacle_map = (gradient_map > 0.25).astype(np.uint8)


fig = px.imshow(obstacle_map)
fig.update_layout(width=1000, height=1000)
fig.show()

In [ ]:
%matplotlib inline
import sys
sys.path.append("..")
from matplotlib import pyplot as plt
from utils.astar import AStarPlanner
astar = AStarPlanner(
    occupancy_grid=obstacle_map,
    meters_per_cell=0.01,
    time_budget=15.0,
    robot_width=0.5, # 0.44
    robot_height=0.9, # 0.88
    step_size=50.0,
    step_size_yaw=30,
    reach_threshold=150.0,
)


# start = (266, 597, np.deg2rad(90))
start = (712, 432, np.deg2rad(90))
# start = (1005, 336, np.deg2rad(90))
goal = (1339, 892, np.deg2rad(0))

vis_img = astar.occupancy_grid.copy()
for point in [start, goal]:
    vis_img, wall = astar.plot_rect(point, vis_img, 100, 3)
plt.imshow(vis_img, cmap='binary')
for x, y, yaw in [start, goal]:
    plt.quiver(x, y, np.cos(yaw), np.sin(yaw),
            angles='xy', scale_units='xy', scale=0.04, color='blue', width=0.01, label='Yaw Direction')
plt.show()


path = astar.plan_and_publish_path(start, goal)

print(f"path: {np.array(path).shape}")
# path = astar.plan(start, goal)
vis_img = astar.occupancy_grid.copy()
for x, y, yaw in astar.path_grid:
    vis_img, wall = astar.plot_rect((x, y, yaw), vis_img, 30, 3)
plt.imshow(vis_img, cmap='binary')
if astar.path_grid:
    path = np.array(astar.path_grid)
    plt.scatter(path[:, 0], path[:, 1], color='red', s=3, label='Planned Path')
    plt.plot(path[:, 0], path[:, 1], 'r-', linewidth=1.5)
    plt.scatter(start[0], start[1], color='blue', s=50, marker='.', label='Start')
    plt.scatter(goal[0], goal[1], color='blue', s=50, marker='*', label='Goal')
    plt.legend()

    x,y,yaw = path[:,0], path[:,1], path[:,2]

    plt.quiver(x, y, np.cos(yaw), np.sin(yaw), np.arange(len(yaw)),
            angles='xy', scale_units='xy', scale=0.04, cmap='viridis', width=0.01, label='Yaw Direction')
    plt.xlabel("X")
    plt.ylabel("Y")
    plt.legend()
    plt.axis("equal")
    plt.title("Waypoints with Yaw Directions in World Frame")
    plt.show()